# Avro Format: a complete Python walkthrough

Apache Avro is a schema-based format for serializing structured data. Serialization converts Python values to bytes; deserialization reconstructs values from bytes.

You will define schemas, validate records, create and store Avro files, read nested data, explore schema evolution, and encode an event for messaging. Run the cells in order in a local Python 3 notebook. No Kafka broker, Snowflake account, cloud storage, or Schema Registry is required.

Generated files are stored in `avro_output` under the kernel working directory. Rerunning the examples replaces their named output files.

## 1. Installation

Use `%pip` to install into the active notebook kernel. The terminal equivalent is `python -m pip install fastavro`. If needed, install Jupyter with `python -m pip install jupyterlab ipykernel`, then launch `python -m jupyterlab`.

We use **fastavro**, a Python Avro implementation. The Apache `avro` package is an alternative; it is not needed here. Restart the kernel after installation if imports fail.

In [ ]:
%pip install fastavro

In [ ]:
from pathlib import Path
from io import BytesIO
from copy import deepcopy
from datetime import date, datetime, timezone
from decimal import Decimal
import json
import fastavro
from fastavro import parse_schema, writer, reader, schemaless_writer, schemaless_reader
from fastavro.validation import validate, ValidationError

OUTPUT_DIR = Path.cwd() / "avro_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("fastavro version:", fastavro.__version__)
print("Output directory:", OUTPUT_DIR.resolve())

## 2. Why Avro and where is it used?

An order service can share a schema with downstream applications so that every consumer interprets its events consistently. Binary serialization avoids repeating JSON field names in every record, and validation detects mismatched values before writing.

| Use case | Role of Avro |
|---|---|
| Kafka events | Encode complete events for producers and consumers using agreed schemas. |
| Change data capture (CDC) | Represent typed database changes and associated metadata. |
| Batch data exchange | Package records and their writer schema in a container file. |
| Data lake ingestion | Preserve complete events before transforming them for analytics. |

JSON is convenient for human inspection. Avro fits exchanging complete typed records. Parquet groups values by column and often suits analytics that scan only selected columns. Size and performance depend on the data, compression, and access pattern.

In a Snowflake pipeline, Avro can be an incoming file format. This walkthrough prepares local files; it does not connect to Snowflake.

## 3. Row format and container structure

Avro binary records store values in schema order, without field names:

```text
record 1: employee_id, name, active, salary, email
record 2: employee_id, name, active, salary, email
```

An Object Container File (OCF) embeds a writer schema in its header and stores records in blocks with synchronization markers. Compression applies to blocks.

```text
employees.avro
  Header: magic + metadata (schema, codec) + sync marker
  Block: record count + block size + encoded records + sync marker
  More blocks as needed
```

A raw serialized datum has no container header; decoding requires a separately supplied writer schema.

Source: [Apache Avro specification](https://avro.apache.org/docs/1.12.0/specification/).

## 4. Native data types

| Primitive | Meaning | Python value |
|---|---|---|
| `null` | No value | `None` |
| `boolean` | True/false | `True` |
| `int` | Signed 32-bit integer | `42` |
| `long` | Signed 64-bit integer | `9000000000` |
| `float` | 32-bit floating point | `1.5` |
| `double` | 64-bit floating point | `99.5` |
| `bytes` | Binary sequence | `b"abc"` |
| `string` | Unicode text | `"Chennai"` |

Python integers must fit the declared range. Python `float` can map to either floating-point type; storing it as Avro `float` can lose precision.

Complex types: **record**, **enum**, **array**, **map**, **union**, and **fixed**. Python representations are dictionaries for records/maps, lists for arrays, strings for enum symbols, exact-length bytes for fixed, and a matching alternative for unions.

Logical types annotate underlying storage: `date` uses `int`, `timestamp-millis` uses `long`, and `decimal` uses `bytes` or `fixed`.

Source: [Avro type definitions](https://avro.apache.org/docs/1.12.0/specification/#primitive-types).

## 5. Create a schema and validate records

A schema is a JSON document. We create a Python dictionary and save it as `.avsc`, the conventional schema-file extension. A schema file contains no employee data.

`type` declares a record, `name` identifies it, `namespace` qualifies its name, and `fields` lists ordered members. `parse_schema` prepares a schema for reuse; `validate` checks a data record.

`["null", "string"]` permits either `None` or text. A default helps a reader fill a field missing from an older writer schema; it does not remove that field from binary encoding. Our input records explicitly provide all fields.

In [ ]:
employee_schema = {
    "type": "record", "name": "Employee", "namespace": "course.avro",
    "doc": "An employee record for learning Avro.",
    "fields": [
        {"name": "employee_id", "type": "long"},
        {"name": "name", "type": "string"},
        {"name": "active", "type": "boolean"},
        {"name": "salary", "type": "double"},
        {"name": "email", "type": ["null", "string"], "default": None},
    ],
}
employees = [
    {"employee_id": 101, "name": "Ananya", "active": True,
     "salary": 75000.0, "email": "ananya@example.com"},
    {"employee_id": 102, "name": "Ravi", "active": False,
     "salary": 68000.0, "email": None},
]
parsed_employee_schema = parse_schema(employee_schema)
for employee in employees:
    assert validate(employee, parsed_employee_schema)
employee_schema_path = OUTPUT_DIR / "employee.avsc"
employee_schema_path.write_text(json.dumps(employee_schema, indent=2), encoding="utf-8")
print(employee_schema_path.read_text(encoding="utf-8"))

### An intentional validation failure

`"101"` is a string, not a `long`. The exception is caught so execution can continue. Correct source values instead of expecting automatic type conversion.

In [ ]:
bad_employee = {**employees[0], "employee_id": "101"}
try:
    validate(bad_employee, parsed_employee_schema)
except ValidationError as exc:
    print("Expected validation error:", exc)
else:
    raise AssertionError("Invalid data should have been rejected")

## 6. Create, store, and read an Avro file

Use binary modes: `wb` creates/replaces a file; `rb` reads one. `writer` accepts an iterable of records. An OCF reader obtains the schema from the header, so reading does not require the separate `.avsc` file.

References: [fastavro writer](https://fastavro.readthedocs.io/en/latest/writer.html), [fastavro reader](https://fastavro.readthedocs.io/en/latest/reader.html).

In [ ]:
employee_path = OUTPUT_DIR / "employees.avro"
with employee_path.open("wb") as file:
    writer(file, parsed_employee_schema, employees, codec="null", validator=True)
with employee_path.open("rb") as file:
    avro_reader = reader(file)
    embedded_schema = avro_reader.writer_schema
    print("Codec:", avro_reader.codec)
    loaded_employees = list(avro_reader)
assert loaded_employees == employees
print("Embedded schema:", json.dumps(embedded_schema, indent=2))
print("Decoded records:", loaded_employees)
with employee_path.open("rb") as file:
    magic = file.read(4)
assert magic == b"Obj\x01"
print("OCF magic:", magic)
print("Stored bytes:", employee_path.stat().st_size)

### Read incrementally

For large files, iterate rather than collecting every row with `list`. The library still buffers blocks, but your application need not retain the whole dataset.

In [ ]:
active_count = 0
with employee_path.open("rb") as file:
    for employee in reader(file):
        if employee["active"]:
            active_count += 1
            print(employee["employee_id"], employee["name"])
assert active_count == 1

## 7. Complete nested order schema

The order combines a nested customer/address, an array of item records, a map of attributes, an enum, a nullable note, and logical types.

```text
OrderEvent
  order_id, order_date, created_at
  customer -> Customer -> address -> Address
  items -> [OrderItem, OrderItem, ...]
  attributes -> {string: string}
  status -> NEW | PAID | CANCELLED
  note -> null | string
```

Use `Decimal` for exact money values. Our price uses precision 10 and scale 2. `fastavro` maps these logical schemas to Python `date`, timezone-aware `datetime`, and `Decimal`. Put `logicalType` inside the field's **type object**.

Source: [fastavro logical types](https://fastavro.readthedocs.io/en/latest/logical_types.html).

In [ ]:
order_schema = {
    "type": "record", "name": "OrderEvent", "namespace": "course.avro",
    "fields": [
        {"name": "order_id", "type": "string"},
        {"name": "order_date", "type": {"type": "int", "logicalType": "date"}},
        {"name": "created_at", "type": {"type": "long", "logicalType": "timestamp-millis"}},
        {"name": "customer", "type": {
            "type": "record", "name": "Customer", "fields": [
                {"name": "customer_id", "type": "long"},
                {"name": "name", "type": "string"},
                {"name": "address", "type": {
                    "type": "record", "name": "Address", "fields": [
                        {"name": "city", "type": "string"},
                        {"name": "postal_code", "type": "string"},
                    ],
                }},
            ],
        }},
        {"name": "items", "type": {"type": "array", "items": {
            "type": "record", "name": "OrderItem", "fields": [
                {"name": "sku", "type": "string"},
                {"name": "quantity", "type": "int"},
                {"name": "unit_price", "type": {
                    "type": "bytes", "logicalType": "decimal", "precision": 10, "scale": 2,
                }},
            ],
        }}},
        {"name": "attributes", "type": {"type": "map", "values": "string"}},
        {"name": "status", "type": {
            "type": "enum", "name": "OrderStatus", "symbols": ["NEW", "PAID", "CANCELLED"],
        }},
        {"name": "note", "type": ["null", "string"], "default": None},
    ],
}
order_schema_path = OUTPUT_DIR / "order_event.avsc"
order_schema_path.write_text(json.dumps(order_schema, indent=2), encoding="utf-8")
# Reload the portable JSON schema, then prepare it for reuse.
parsed_order_schema = parse_schema(json.loads(order_schema_path.read_text(encoding="utf-8")))
print(json.dumps(order_schema, indent=2))

### Create nested Python records

Map keys are strings. Each item follows the same `OrderItem` schema. Postal codes are strings to preserve leading zeros. The timestamps have UTC time zones and millisecond-compatible precision.

In [ ]:
orders = [
    {
        "order_id": "ORD-1001", "order_date": date(2026, 9, 1),
        "created_at": datetime(2026, 9, 1, 10, 30, tzinfo=timezone.utc),
        "customer": {"customer_id": 501, "name": "Meera",
                     "address": {"city": "Chennai", "postal_code": "600001"}},
        "items": [
            {"sku": "BOOK-01", "quantity": 2, "unit_price": Decimal("499.50")},
            {"sku": "PEN-01", "quantity": 3, "unit_price": Decimal("25.00")},
        ],
        "attributes": {"channel": "mobile", "currency": "INR"},
        "status": "PAID", "note": None,
    },
    {
        "order_id": "ORD-1002", "order_date": date(2026, 9, 2),
        "created_at": datetime(2026, 9, 2, 9, 0, tzinfo=timezone.utc),
        "customer": {"customer_id": 502, "name": "Arjun",
                     "address": {"city": "Pune", "postal_code": "411001"}},
        "items": [{"sku": "BAG-01", "quantity": 1, "unit_price": Decimal("1250.00")}],
        "attributes": {"channel": "web", "currency": "INR"},
        "status": "NEW", "note": "Deliver after 6 PM",
    },
]
for order in orders:
    assert validate(order, parsed_order_schema)
print("Validated", len(orders), "nested orders")

### Write compressed orders, read them back, and calculate totals

The `deflate` codec compresses record blocks without requiring an extra codec package here. Equality checks verify that nested structures and logical values survive the round trip.

In [ ]:
orders_path = OUTPUT_DIR / "orders.avro"
with orders_path.open("wb") as file:
    writer(file, parsed_order_schema, orders, codec="deflate", validator=True)
with orders_path.open("rb") as file:
    loaded_orders = list(reader(file))
assert loaded_orders == orders
for order in loaded_orders:
    total = sum((item["quantity"] * item["unit_price"] for item in order["items"]), Decimal("0.00"))
    print(order["order_id"], order["customer"]["address"]["city"], total,
          order["attributes"]["currency"], order["created_at"].isoformat())
assert isinstance(loaded_orders[0]["items"][0]["unit_price"], Decimal)
assert isinstance(loaded_orders[0]["order_date"], date)

## 8. Compare compression on identical data

Both files contain the same schema and records. Sizes include headers and block overhead. Compression costs CPU and may not shrink tiny files; this is an illustration, not a benchmark.

In [ ]:
plain_orders_path = OUTPUT_DIR / "orders_uncompressed.avro"
with plain_orders_path.open("wb") as file:
    writer(file, parsed_order_schema, orders, codec="null", validator=True)
for path in [plain_orders_path, orders_path]:
    print(f"{path.name}: {path.stat().st_size:,} bytes")
    with path.open("rb") as file:
        assert list(reader(file)) == orders

## 9. Exercise every primitive type, plus fixed

This example fills the remaining gaps from the type table. Exactly representable floating-point values make equality checks straightforward. Unlike `bytes`, `fixed` requires the byte length declared in the schema.

In [ ]:
types_schema = parse_schema({
    "type": "record", "name": "TypeDemo", "namespace": "course.avro",
    "fields": [
        {"name": "empty", "type": "null"},
        {"name": "enabled", "type": "boolean"},
        {"name": "count", "type": "int"},
        {"name": "large_count", "type": "long"},
        {"name": "ratio", "type": "float"},
        {"name": "measurement", "type": "double"},
        {"name": "payload", "type": "bytes"},
        {"name": "label", "type": "string"},
        {"name": "tag", "type": {"type": "fixed", "name": "FourBytes", "size": 4}},
    ],
})
type_record = {"empty": None, "enabled": True, "count": 42, "large_count": 9000000000,
               "ratio": 1.5, "measurement": 2.25, "payload": b"abc",
               "label": "Chennai", "tag": b"ABCD"}
assert validate(type_record, types_schema)
types_path = OUTPUT_DIR / "types.avro"
with types_path.open("wb") as file:
    writer(file, types_schema, [type_record], validator=True)
with types_path.open("rb") as file:
    decoded = next(reader(file))
assert decoded == type_record
for name, value in decoded.items():
    print(name, repr(value), type(value).__name__)

## 10. Schema evolution: a new reader, old data

Add a `department` field with a default. The reader supplies it for old records. The original file and its writer schema remain unchanged.

Compatibility is directional. Reading old data with a new schema does not prove every old consumer can read all new data. Test the actual writer/reader pairs before application changes.

Reference: [fastavro reader schema](https://fastavro.readthedocs.io/en/latest/reader.html).

In [ ]:
employee_schema_v2 = deepcopy(employee_schema)
employee_schema_v2["fields"].append(
    {"name": "department", "type": "string", "default": "UNKNOWN"}
)
with employee_path.open("rb") as file:
    evolved_employees = list(reader(file, reader_schema=employee_schema_v2))
assert all(row["department"] == "UNKNOWN" for row in evolved_employees)
assert all("department" not in row for row in loaded_employees)
print(evolved_employees)

## 11. Kafka-style payloads without a broker or registry

Kafka transports bytes; applications choose their serialization. We simulate a producer and consumer sharing a schema. No message is sent over a network.

`schemaless_writer` means no container header, **not** no schema. Without a registry, distribute versioned writer schemas with applications and agree on a version identifier in message headers or an envelope.

These raw bytes are neither an OCF file nor Avro single-object framing nor Confluent serializer framing. Producers and consumers must agree on their protocol.

References: [schemaless writer](https://fastavro.readthedocs.io/en/latest/writer.html#fastavro.write.schemaless_writer), [schemaless reader](https://fastavro.readthedocs.io/en/latest/reader.html#fastavro.read.schemaless_reader).

In [ ]:
# Simulated producer: encode a single event with the agreed writer schema.
message_buffer = BytesIO()
schemaless_writer(message_buffer, parsed_order_schema, orders[0])
message_value = message_buffer.getvalue()
message_path = OUTPUT_DIR / "order_message.bin"
message_path.write_bytes(message_value)

# Simulated consumer: load the same schema and decode the received bytes.
consumer_schema = parse_schema(json.loads(order_schema_path.read_text(encoding="utf-8")))
received = schemaless_reader(BytesIO(message_path.read_bytes()), consumer_schema)
assert received == orders[0]
print("Event bytes:", len(message_value))
print("Received order:", received["order_id"])

### Optional Schema Registry and payload size

A registry can manage schema versions and compatibility. With Confluent's schema-ID wire format, a message carries a small identifier instead of a full schema; consumers resolve and cache the schema. This reduces payload size **compared with embedding the whole schema in each message**.

Our raw event already excludes the schema, so adding registry framing would not make those bytes smaller. OCF files already share a header schema across records. No registry installation or configuration is needed for this notebook.

The measurement below compares raw bytes with a hypothetical repeated-schema envelope, before envelope overhead. It does not implement Kafka or registry framing.

Source: [Confluent serialization formats](https://docs.confluent.io/platform/current/schema-registry/fundamentals/serdes-develop/index.html).

In [ ]:
schema_bytes = json.dumps(order_schema, separators=(",", ":")).encode("utf-8")
print("Raw event:", len(message_value), "bytes")
print("Full schema plus event:", len(schema_bytes) + len(message_value), "bytes")
print("Avoided schema repetition per event:", len(schema_bytes), "bytes")

## 12. Inspect the stored artifacts

| File | Purpose |
|---|---|
| `employee.avsc` | Employee JSON schema |
| `employees.avro` | Employee container file |
| `order_event.avsc` | Complete nested JSON schema |
| `orders.avro` | Deflate-compressed nested orders |
| `orders_uncompressed.avro` | Same orders without compression |
| `types.avro` | Primitive and fixed examples |
| `order_message.bin` | Raw event requiring an external writer schema |

Open `.avsc` files as text. Inspect `.avro` files with an Avro reader. Container files can be transferred to other machines and read by compatible Avro implementations.

In [ ]:
artifacts = [employee_schema_path, employee_path, order_schema_path, orders_path,
             plain_orders_path, types_path, message_path]
for path in artifacts:
    assert path.is_file()
    print(f"{path.name:28} {path.stat().st_size:>7,} bytes")
print("All walkthrough round-trip checks passed.")

## 13. Practice and troubleshooting

1. Add another order with two items, validate it, rewrite the file, and check the record count.
2. Replace a quantity with `"two"`, catch the validation error, and correct it.
3. Add nullable `phone` to a new employee reader schema with a `None` default and read the old file.
4. Explain why `orders.avro` is readable without its `.avsc`, but `order_message.bin` needs a schema.
5. Compare compression on 1,000 representative orders.

Common issues:

- **Import failure:** install into the selected notebook kernel and restart if needed.
- **Validation error:** inspect the field path, type, enum symbol, or fixed-byte length.
- **Decimal error:** use `Decimal("499.50")` and stay within precision and scale.
- **Binary decoding error:** choose `reader` for OCF or `schemaless_reader` with a schema for raw data.
- **Unexpected file location:** check the printed `OUTPUT_DIR`; the kernel working directory may differ from the notebook folder.
- **Schema mismatch:** retain writer schema versions and test compatibility before deployment.